In [2]:
import numpy as np, pandas as pd

In [ ]:
data = {
    "body_weight": [
        184.5, 183.5, 184.5, 184.5, 186.5,
        184.7, 184.7, 185.2, 186.6, 188.7,
        186.6, 187.0, 185.3, 186.5
    ],
    "datetime": pd.to_datetime([
        "2026-05-13",
        "2026-05-10",
        "2026-05-08",
        "2026-05-07",
        "2026-05-06",
        "2026-05-03",
        "2026-05-02",
        "2026-05-01",
        "2026-04-30",
        "2026-04-29",
        "2026-04-28",
        "2026-04-27",
        "2026-04-26",
        "2026-04-25"
    ])
}

df = pd.DataFrame(data)
df

In [ ]:
# get dynamic delta T
df["delta_v"] = (
    (df["datetime"] - df["datetime"].shift(-1)).dt.days
)
# df['delta'] = df['datetime'].dt.day

## State Prediction

$$
\hat{x}_k = A x_{k-1} + B u_k + w_k
$$

Where:
- $A$ = state transition matrix (models how the state naturally evolves from one time step to the next)
- $\hat{x}_{k|k-1}$ = predicted state estimate at time step $k$
- $F_k$ = state transition matrix
- $\hat{x}_{k-1|k-1}$ = previous corrected state estimate
- $B_k$ = control-input matrix
- $u_k$ = control vector
- $w_k$ = process noise vector (random uncertainty/error in the system dynamics)

In [ ]:
dt = 1
X = np.array([
    [4000],
    [280]
], dtype=float)
A = np.array([
    [1, dt],
    [0, 1]
], dtype=float)
B = np.array([
    [0.5*dt**2],
    [dt]
], dtype=float)
u = np.array([
    [2]
], dtype=float)
w = np.zeros((2, 1), dtype=float)

predicted_state = A@X + B@u + w
predicted_state

## Kalman Filter: Process Covariance Prediction

$$
P_k = A P_{k-1} A^T + q
$$

Where:
- $P_k$ is the predicted covariance matrix
- $A$ is the state transition matrix
- $P_{k-1}$ is the previous covariance matrix
- $q$ is the process noise covariance

In [ ]:
# process errors
err_proc_pos = 20
err_proc_vel = 5

# at first this will be the initial PCM
P = np.array([
    [err_proc_pos**2, err_proc_pos*err_proc_vel],
    [err_proc_vel*err_proc_pos, err_proc_vel**2]
], dtype=float)
P[0][1], P[1][0] = 0, 0

# process noise/uncertainty
#* shape must match the predicted PCM output shape and the output of the A@P@A.T operation (eg. 2x2)
q = np.zeros((2, 2), dtype=float)

# equation
predicted_pcm = A@P@A.T + q
predicted_pcm[0][1], predicted_pcm[1][0] = 0, 0
P, predicted_pcm

## Kalman Filter: Kalman Gain Equation

$$
K_k = P_k H^T \left(H P_k H^T + R\right)^{-1}
$$

Where:
- $K_k$ is the Kalman Gain at time step $k$
- $P_k$ is the predicted state covariance matrix ($predicted_pcm)
- $H$ is the observation (measurement) matrix
- $H^T$ is the transpose of the observation matrix
- $R$ is the observation noise covariance matrix
- $\left(H P_k H^T + R\right)$ is the innovation (residual) covariance matrix
- $\left(H P_k H^T + R\right)^{-1}$ is the inverse of the innovation covariance

In [ ]:
# H/C shape = (num_measurements, num_states) (ussually H and C are one in the same)
# I (all identity matrices) = (num_states, num_states)

# observation erros/noise
err_obs_pos = 25 # meters
err_obs_vel = 6 #m/s

# initialize observation matrix (Somestimes C, shape must match the PCM)
H = np.array([
    [1, 0],
    [0, 1]
], dtype=float)

# eg. H matrix of measuremt just position in x and state of both position and velocity in X (meaning consider all of the position in X and none of velcity in X since we dont measure it)
# H = np.array([
#     [1, 0]
# ], dtype=float)

# observation errors (have option to use first n observations as batch to calcaulte the above, can also choose to 0 out the covariances)
R = np.array([
    [err_obs_pos**2, err_obs_pos*err_obs_vel],
    [err_obs_vel*err_obs_pos, err_obs_vel**2]
], dtype=float)
R = np.array([
    [err_obs_pos**2, 0.0],
    [0.0, err_obs_vel**2]
], dtype=float)

# equation (deprecated, we dont actually need to calculate the inverse)
# S = H@predicted_pcm@H.T + R # inovation covariance
# K = predicted_pcm@H.T@np.linalg.inv(S)

'''
Note: np.linalg.solve(A, B) only solves equation in the form: AX = B
'''
S = H@predicted_pcm@H.T + R #inovation covariance
K = np.linalg.solve(S.T, (H@predicted_pcm.T)).T
K

## Observation / Measurement Update

$$
y_k^{updated} = C Y_k + z_k
$$

Where:
- $y_k^{updated}$ is the updated observation vector
- $C$ or $H$ is the observation matrix
- $Y_k$ is the raw measurement vector
- $z_k$ is the observation noise (error) vector

In [ ]:
# eg. C matrix (observation matrix) just position in x and state of both position and velocity in X (meaning consider all of the position in X and none of velcity in X since we dont measure it)
# C = np.array([
#     [1, 0]
# ], dtype=float)

# new observations
Y = np.array([
    [4260],
    [282]
], dtype=float)

# measurement error/noise vector
z = np.array([
    [0], # 3 meters
    [0]  # 2 m/s
], dtype=float)

# equation
updated_observation = H@Y + z # almost always set to 0
updated_observation

## Kalman Filter: State Update Equation

$$
\hat{x}_k = \hat{x}_{k|k-1} + K_k \left(y_k - H \hat{x}_{k|k-1}\right)
$$

Where:
- $\hat{x}_k$ is the updated (posterior) state estimate at time step $k$
- $\hat{x}_{k|k-1}$ is the predicted (prior) state estimate ($predicted_state)
- $K_k$ is the Kalman Gain
- $y_k$ is the observed measurement vector ($updated_observation)
- $H$ is the observation (measurement) matrix
- $H \hat{x}_{k|k-1}$ is the predicted measurement from the prior state
- $(y_k - H \hat{x}_{k|k-1})$ is the innovation (measurement residual)

In [ ]:
# equation
X_hat = predicted_state + K@(updated_observation - H@predicted_state)
X_hat

## Combined state update formula

$$
\hat{x}_k = \hat{x}_{k|k-1} + K_k \left(C Y_k + z_k - H \hat{x}_{k|k-1}\right)
$$

## Kalman Filter: Covariance Update Equation

$$
P_k = \left(I - K_k H\right) P_{k|k-1}
$$

Where:
- $P_k$ is the updated (posterior) state covariance matrix
- $P_{k|k-1}$ is the predicted (prior) covariance matrix ($predicted_pcm)
- $K_k$ is the Kalman Gain
- $H$ is the observation (measurement) matrix
- $I$ is the identity matrix (same dimension as the state covariance matrix)
- $(I - K_k H)$ is the uncertainty reduction factor applied after measurement update

In [ ]:
I = np.identity(2)

# equation
P = (I - K@H)@predicted_pcm  # faster (asymetrical)
sod = (I - K@H)@predicted_pcm@(I - K@H).T + K@R@K.T  # Joseph form (symetrical) 